## **Title**

In [1]:
#News Topic Classifier Using BERT
#Problem Statement & Objective

#The objective of this task is to build a text classification model using a transformer (BERT) to classify news headlines into categories:

#World
#Sports
#Business
#Sci/Tech

#We will:

#Load and preprocess dataset
#Tokenize text using BERT tokenizer
#Fine-tune BERT model
#Evaluate using Accuracy & F1-score
#Deploy using Gradio

## **Import Libraries**

In [2]:
# ==============================
# IMPORT REQUIRED LIBRARIES
# ==============================
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import accuracy_score, f1_score

## **Dataset Loading**

In [3]:
# ==============================
# LOAD AG NEWS DATASET
# ==============================
dataset = load_dataset("ag_news")

# Train and test split
train_dataset = dataset["train"]
test_dataset = dataset["test"]

print(train_dataset[0])

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


## **LOAD TOKENIZER (BERT)**

In [4]:
# ==============================
# LOAD BERT TOKENIZER
# ==============================
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

## **DATA PREPROCESSING / TOKENIZATION**

In [5]:
# ==============================
# TOKENIZATION FUNCTION
# ==============================
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length = 64)

# Apply tokenization
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

In [6]:
train_dataset = train_dataset.select(range(3000))
test_dataset = test_dataset.select(range(1000))

## **FORMAT DATA FOR PYTORCH**

In [7]:
# ==============================
# SET FORMAT FOR PYTORCH
# ==============================
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

## **LOAD BERT MODEL**

In [8]:
# ==============================
# LOAD PRETRAINED BERT MODEL
# ==============================
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4   # AG News has 4 categories
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## **DEFINE EVALUATION METRICS (ACCURACY + F1)**

In [9]:
# ==============================
# METRICS FUNCTION
# ==============================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")

    return {
        "accuracy": acc,
        "f1": f1
    }

## **TRAINING ARGUMENTS**

In [10]:

# ==============================
# FAST TRAINING SETTINGS 
# ==============================

num_train_epochs = 1

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=4,   # reduce for CPU
    per_device_eval_batch_size=4,

    num_train_epochs=1,

    weight_decay=0.01,

    logging_steps=50,

    load_best_model_at_end=True
)

## **TRAINER SETUP**

In [11]:
# ==============================
# TRAINER INITIALIZATION
# ==============================
import os
os.environ["DISABLE_TQDM"] = "1"
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[]
)

## **TRAIN MODEL**

In [12]:
# ==============================
# START TRAINING
# ==============================
trainer.train()

C:\Users\Zaman Computer\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.529128,0.443683,0.890000,0.888886


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=750, training_loss=0.5378487218221029, metrics={'train_runtime': 2100.0816, 'train_samples_per_second': 1.429, 'train_steps_per_second': 0.357, 'total_flos': 98668417536000.0, 'train_loss': 0.5378487218221029, 'epoch': 1.0})

## **EVALUATE MODEL**

In [14]:
predictions = trainer.predict(test_dataset)

import numpy as np
from sklearn.metrics import accuracy_score, f1_score

preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

print("Accuracy:", accuracy_score(labels, preds))
print("F1 Score:", f1_score(labels, preds, average="weighted"))

C:\Users\Zaman Computer\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Accuracy: 0.89
F1 Score: 0.8888859563812072


## **SAVE MODEL**

In [15]:
# ==============================
# SAVE MODEL & TOKENIZER
# ==============================
model.save_pretrained("news_classifier_model")
tokenizer.save_pretrained("news_classifier_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('news_classifier_model\\tokenizer_config.json',
 'news_classifier_model\\tokenizer.json')

## **STREAMLIT APP (DEPLOYMENT)**

In [16]:
# ==============================
# STREAMLIT APP FOR LIVE PREDICTION
# ==============================

import streamlit as st
import torch
from transformers import BertTokenizer, BertForSequenceClassification

# Load model
model = BertForSequenceClassification.from_pretrained("news_classifier_model")
tokenizer = BertTokenizer.from_pretrained("news_classifier_model")

# Class labels (AG News)
labels = ["World", "Sports", "Business", "Sci/Tech"]

# Title
st.title("News Topic Classifier (BERT)")

# Input box
text = st.text_area("Enter News Headline")

if st.button("Predict"):
    # Tokenize input
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # Prediction
    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=1).item()

    st.success(f"Predicted Category: {labels[prediction]}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-04-28 16:38:46.297 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-28 16:38:46.456 
  command:

    streamlit run C:\Users\Zaman Computer\AppData\Roaming\Python\Python313\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-04-28 16:38:46.457 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-28 16:38:46.458 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-28 16:38:46.459 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-28 16:38:46.460 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-28 16:38:46.461 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-28 16:38:46

## **Final Summary / Insights**

### How BERT works for text classification
### Tokenization & padding
### Fine-tuning transformer models
### Evaluation (Accuracy + F1-score)
### Model deployment using Streamlit